# Interactive map

Goal: Implement an interactive map that allows filtering of projects by type (funding_scheme), by total cost, by type, by field / subfield / niche, startyear of the project.

We create several interactive maps with plotly:
- Bubble chart: country-specific bubbble
    - Bubble magnitude needs a metric
    - Metric: total contribution / total cost / number of projects (normalized by number of inhabitants) / number of publications /100k money received
    - Add colormap right to the plot that colorcodes the encoded metric
    - Allow user to click on bubble and then zoom in one layer, on the country user clicked on. Then recreate the same map, but this time on the level of cities. 
Requirements:
- `plotly`: creating interactive plots
- `pandas`: data management
- `numpy`: 
- `folium`

In [1]:
# imports
import numpy as np
import pandas as pd
import plotly.express as px
import os
import sys
from pathlib import Path

from ipywidgets import interact, widgets
from IPython.display import display, clear_output

import plotly.io as pio
pio.renderers.default = 'notebook' 

## Load data

In [2]:
project_root = Path.cwd().parent  # assumes you're in /notebooks
sys.path.append(str(project_root))

from backend.etl.ingestion import inspect_bad_lines, auto_fix_row, robust_csv_reader

In [9]:
# get projects and organizations
notebook_dir = os.getcwd()
base_dir = os.path.dirname(notebook_dir)

df_proj = robust_csv_reader(f'{base_dir}/data/processed/project_df.csv', delimiter=',')
# df_proj_interim = robust_csv_reader(f'{base_dir}/data/interim/project_df.csv', delimiter=',')
df_org = robust_csv_reader(f'{base_dir}/data/processed/organization_df.csv', delimiter=',')
df_topics = robust_csv_reader(f'{base_dir}/data/processed/topics_df.csv', delimiter=',')

In [10]:
list(df_proj_processed.keys())

['id',
 'acronym',
 'status',
 'title',
 'startDate',
 'endDate',
 'totalCost',
 'ecMaxContribution',
 'legalBasis',
 'topics',
 'ecSignatureDate',
 'frameworkProgramme',
 'masterCall',
 'subCall',
 'fundingScheme',
 'nature',
 'objective',
 'contentUpdateDate',
 'rcn',
 'grantDoi',
 'duration_days',
 'duration_months',
 'duration_years',
 'projectID_x',
 'n_institutions',
 'projectID_y',
 'institutions',
 'projectID',
 'coordinator_name',
 'ecContribution_per_year',
 'totalCost_per_year',
 'field_class',
 'field',
 'subfield',
 'niche']

In [11]:
# change the keys:
columns_renamed = {
    'status': 'status',
    'title': 'title',
    'startDate': 'start_date',
    'endDate': 'end_date',
    'totalCost': 'total_cost',
    'ecMaxContribution': 'ec_max_contribution',
    'ecSignatureDate': 'ec_signature_date',
    'frameworkProgramme': 'framework_programme',
    'masterCall': 'master_call',
    'subCall': 'sub_call',
    'fundingScheme': 'funding_scheme',
    'nature': 'nature',
    'objective': 'objective',
    'contentUpdateDate': 'content_update_date',
    'rcn': 'rcn',
    'grantDoi': 'grant_doi',
    'duration_days': 'duration_days',
    'duration_months': 'duration_months',
    'duration_years': 'duration_years',
    'n_institutions': 'n_institutions',
    'coordinator_name': 'coordinator_name',
    'ecContribution_per_year': 'ec_contribution_per_year',
    'totalCost_per_year': 'total_cost_per_year',
    'field_class': 'field_class',
    'field': 'field',
    'subfield': 'sub_field',
    'niche': 'niche'
}

df_proj = df_proj.rename(columns=columns_renamed)

In [12]:
org_column_renaming = {
    'projectID': 'id',
    'projectAcronym': 'project_acronym',
    'organisationID': 'organisation_id',
    'vatNumber': 'vat_number',
    'name': 'name',
    'shortName': 'short_name',
    'SME': 'sme',
    'activityType': 'activity_type',
    'street': 'street',
    'postCode': 'post_code',
    'city': 'city',
    'country': 'country',
    'nutsCode': 'nuts_code',
    'geolocation': 'geolocation',
    'organizationURL': 'organization_url',
    'contactForm': 'contact_form',
    'contentUpdateDate': 'content_update_date',
    'rcn': 'rcn',
    'order': 'order',
    'role': 'role',
    'ecContribution': 'ec_contribution',
    'netEcContribution': 'net_ec_contribution',
    'totalCost': 'total_cost',
    'endOfParticipation': 'end_of_participation',
    'active': 'active'
}

df_org = df_org.rename(columns=org_column_renaming)


In [83]:
df_org['geolocation']

0                        -18.7920779,47.7823214
1                          14.693425,-17.447938
2                                     9.0,38.75
3                         51.5207316,-0.1294867
4                   46.221901,6.148001732215738
                          ...                  
101148    50.341363650000005,19.284806373671188
101149                      45.020781,7.8338801
101150     50.829915150000005,4.350155933524583
101151                     45.4698524,9.2027289
101152                    38.3964248,-0.5250339
Name: geolocation, Length: 101153, dtype: object

In [81]:
df_org.head(5)

,id,project_acronym,organisation_id,vat_number,name,short_name,sme,activity_type,street,post_code,...,contact_form,content_update_date,rcn,order,role,ec_contribution,net_ec_contribution,total_cost,end_of_participation,active
0,101159220,PvSeroRDT,986872084,,Institut Pasteur de Madagascar,IPM,False,REC,Ambatofotsikely - Avaradoha,101,...,https://ec.europa.eu/info/funding-tenders/oppo...,2024-12-24 11:18:48,1947090,4,associatedPartner,,0.0,0,False,
1,101159220,PvSeroRDT,999542806,,INSTITUT PASTEUR DE DAKAR,,False,REC,AVENUE PASTEUR 36,DAKAR,...,https://ec.europa.eu/info/funding-tenders/oppo...,2024-12-24 11:18:48,1906512,2,participant,1777625.0,1777625.0,1777625,False,
2,101159220,PvSeroRDT,889740358,,ARMAUER HANSEN RESEARCH INSTITUTE,,False,REC,JIMMA ROAD ALERT COMPOUND,1005,...,https://ec.europa.eu/info/funding-tenders/oppo...,2024-12-24 11:18:48,1975141,3,participant,272174.38,272174.38,"272174,38",False,
3,101159220,PvSeroRDT,999912667,GB233756066,LONDON SCHOOL OF HYGIENE AND TROPICAL MEDICINE...,LSHTM,False,HES,KEPPEL STREET,WC1E 7HT,...,https://ec.europa.eu/info/funding-tenders/oppo...,2024-12-24 11:18:48,1906028,6,associatedPartner,,0.0,0,False,
4,101159220,PvSeroRDT,999679964,,FOUNDATION FOR INNOVATIVE NEW DIAGNOSTICS,"FIND, the global alliance for diagnostics",False,OTH,"CAMPUS BIOTECH, CHEMIN DES MINES 9",1202,...,https://ec.europa.eu/info/funding-tenders/oppo...,2024-12-24 11:18:48,1906308,7,associatedPartner,,0.0,0,False,


## Start creating the plot
We implement the features implemented above. 

In [13]:
df_org['country'].unique()

array(['MG', 'SN', 'ET', 'UK', 'CH', 'FR', 'AU', 'FI', 'DK', 'ES', 'SI',
       'LT', 'PL', 'NL', 'PT', 'BE', 'DE', 'US', 'NO', 'TR', 'ZA', 'ZM',
       'ZW', 'CI', 'SK', 'BG', 'RO', 'EL', 'IL', 'IT', 'EE', 'IE', 'HU',
       'CZ', 'AT', 'LV', 'UA', 'GN', 'ML', 'SE', 'BW', 'MZ', 'LS', '',
       'SZ', 'BF', 'GH', 'CY', 'MT', 'CM', 'LU', 'NG', 'TZ', 'MW', 'UG',
       'KE', 'CN', 'IN', 'KR', 'RS', 'EG', 'AR', 'HR', 'AM', 'BR', 'CV',
       'CA', 'TN', 'AO', 'ST', 'CO', 'BT', 'PY', 'CF', 'DZ', 'GQ', 'LK',
       'CL', 'AL', 'IS', 'CD', 'BI', 'MX', 'ME', 'MN', 'TH', 'KZ', 'JP',
       'VA', 'NZ', 'EC', 'MD', 'UZ', 'AZ', 'SG', 'PK', 'TW', 'GU', 'CR',
       'PE', 'LB', 'BA', 'MA', 'VN', 'MK', 'BJ', 'GA', 'MY', 'XK', 'PS',
       'PH', 'SA', 'RW', 'ID', 'FO', 'CU', 'KG', 'BD', 'PF', 'LR', 'SL',
       'VE', 'GE', 'JO', 'FJ', 'UY', 'CG', 'AF', 'IQ', 'HK', 'TJ', 'TM',
       'BO', 'MV', 'IM', 'NP', 'MH', 'AD', 'MU', 'PA', 'DJ', 'TD', 'BQ',
       'AW', 'GM', 'MR', 'TG', 'SD', 'PG', 'LA', 'MO'

Create dictionary with country-specific population numbers. Generated by ChatGPT so take care with those values

In [14]:
country_populations = {
    'MG': 29452714, 'SN': 18847519, 'ET': 118550298, 'UK': 68459055, 'CH': 8860574, 'FR': 68374591,
    'AU': 26768598, 'FI': 5626414, 'DK': 5973136, 'ES': 47280433, 'SI': 2097893, 'LT': 2628186,
    'PL': 38746310, 'NL': 17772378, 'PT': 10207177, 'BE': 11977634, 'DE': 84119100, 'US': 347275807,
    'NO': 5509733, 'TR': 84119531, 'ZA': 60442647, 'ZM': 20799116, 'ZW': 17150352, 'CI': 29981758,
    'SK': 5563649, 'BG': 6782659, 'RO': 18148155, 'EL': 10461091, 'IL': 9402617, 'IT': 60964931,
    'EE': 1193791, 'IE': 5233461, 'HU': 9855745, 'CZ': 10837890, 'AT': 8967982, 'LV': 1801246,
    'UA': 35661826, 'GN': 13986179, 'ML': 21990607, 'SE': 10589835, 'BW': 2450668, 'MZ': 33350954,
    'LS': 2227548, 'SZ': 1138089, 'BF': 23042199, 'GH': 34589092, 'CY': 1320525, 'MT': 469730,
    'CM': 30966105, 'LU': 671254, 'NG': 237527782, 'TZ': 67462121, 'MW': 21763309, 'UG': 49283041,
    'KE': 58246378, 'CN': 1416096094, 'IN': 1463865525, 'KR': 52081799, 'RS': 6652212,
    'EG': 111247248, 'AR': 46994384, 'HR': 4150116, 'AM': 2976765, 'BR': 212812405, 'CV': 611014,
    'CA': 38794813, 'TN': 12048847, 'AO': 37202061, 'ST': 223561, 'CO': 49588357, 'BT': 884546,
    'PY': 7522549, 'CF': 5650957, 'DZ': 47022473, 'GQ': 1795834, 'LK': 21982608, 'CL': 18664652,
    'AL': 3107100, 'IS': 364036, 'CD': 115403027, 'BI': 13590102, 'MX': 130739927, 'ME': 599849,
    'MN': 3281676, 'TH': 69920998, 'KZ': 20260006, 'JP': 123201945, 'VA': 496, 'NZ': 5161211,
    'EC': 18309984, 'MD': 3599528, 'UZ': 36520593, 'AZ': 10650239, 'SG': 6028459, 'PK': 255219554,
    'TW': 23595274, 'GU': 169532, 'CR': 5265575, 'PE': 32600249, 'LB': 5364482, 'BA': 3798671,
    'MA': 37387585, 'VN': 105758975, 'MK': 2135622, 'BJ': 14697052, 'GA': 2455105, 'MY': 34564810,
    'XK': 1977093, 'PS': 3243369, 'PH': 118277063, 'SA': 36544431, 'RW': 13623302, 'ID': 285721236,
    'FO': 52933, 'CU': 10966038, 'KG': 6172101, 'BD': 175686899, 'PF': 303540, 'LR': 5437249,
    'SL': 9121049, 'VE': 31250306, 'GE': 4900961, 'JO': 11174024, 'FJ': 951611, 'UY': 3425330,
    'CG': 6097665, 'AF': 40121552, 'IQ': 42083436, 'HK': 7297821, 'TJ': 10394063, 'TM': 5744151,
    'BO': 12311974, 'MV': 388858, 'IM': 92269, 'NP': 31122387, 'MH': 82011, 'AD': 85370,
    'MU': 1310504, 'PA': 4470241, 'DJ': 994974, 'TD': 19093595, 'BQ': 25519, 'AW': 125063,
    'GM': 2523327, 'MR': 4328040, 'TG': 8917994, 'SD': 50467278,
    '': None  # No country code provided
}


In [15]:
print(df_org.columns)

Index(['id', 'project_acronym', 'organisation_id', 'vat_number', 'name',
       'short_name', 'sme', 'activity_type', 'street', 'post_code', 'city',
       'country', 'nuts_code', 'geolocation', 'organization_url',
       'contact_form', 'content_update_date', 'rcn', 'order', 'role',
       'ec_contribution', 'net_ec_contribution', 'total_cost',
       'end_of_participation', 'active'],
      dtype='object')


In [41]:
from backend.etl.cleaning import clean_date_column
df_proj['start_year'] = clean_date_column(df_proj['start_date']).dt.year

# STEP 1: Get project-country combinations (one row per country per project)
proj_country = df_org[['id', 'country']]

# add contribution per project
proj_country = proj_country.merge(
    df_proj[['id', 'ec_max_contribution']],
    on='id',
    how='left'
)

proj_country

/Users/bertdepoorter/Nextcloud/EU_Horizon_Dashboard/backend/etl/cleaning.py:49: UserWarning:

The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.



,id,country,ec_max_contribution
0,101159220,MG,
1,101159220,SN,
2,101159220,ET,
3,101159220,UK,
4,101159220,CH,
...,...,...,...
101148,101172981,PL,999500.0
101149,101131799,IT,1046625.0
101150,101131799,BE,1046625.0
101151,101131799,IT,1046625.0


In [42]:
try:
    assert True in proj_country['ec_max_contribution'].isna()
except AssertionError:
    print('AssertionError: no NaN values in ec_max_contribution')

AssertionError: no NaN values in ec_max_contribution


In [43]:
# set column ec-max_contribution to numeric
proj_country['ec_max_contribution'] = pd.to_numeric(proj_country['ec_max_contribution'], errors='coerce').fillna(0.0)
proj_country['ec_max_contribution'].unique()

array([      0.,  500000., 6771571., ...,  850000.,  999500., 1046625.],
      shape=(4905,))

In [48]:
# Calculate total contribution per country (sum over all projects related to that country)
country_data = proj_country.groupby('country')['ec_max_contribution'].sum().reset_index()

# Rename column for clarity
country_data = country_data.rename(columns={'ec_max_contribution': 'total_contribution_country'})

# Merge back into proj_country on 'country'
country_data = country_data.merge(proj_country, on='country', how='left')
country_data

,country,total_contribution_country,id,ec_max_contribution
0,,16876536.0,101103195,0.0
1,,16876536.0,101081581,1881600.0
2,,16876536.0,101103174,0.0
3,,16876536.0,101063992,0.0
4,,16876536.0,101065058,0.0
...,...,...,...,...
101148,ZW,4989715.0,101103281,0.0
101149,ZW,4989715.0,101145811,0.0
101150,ZW,4989715.0,101137814,4989715.0
101151,ZW,4989715.0,101145636,0.0


In [51]:
# remove redundant rows
country_data = country_data[['country', 'total_contribution_country']].drop_duplicates()
country_data

,country,total_contribution_country
0,,16876536.0
13,AD,0.0
14,AE,8269188.0
19,AF,3392025.0
20,AI,0.0
...,...,...
100892,VN,26396005.0
100910,XK,32109580.0
100926,ZA,482231539.0
101126,ZM,43828596.0


In [54]:
# STEP 1: Project-country combinations
proj_country = df_org[['id', 'country']].drop_duplicates()

# STEP 2: Merge EC max contribution per project into proj_country
proj_country = proj_country.merge(
    df_proj[['id', 'ec_max_contribution']],
    on='id',
    how='left'
)

# STEP 3: Ensure ec_max_contribution is numeric and handle missing values
proj_country['ec_max_contribution'] = pd.to_numeric(
    proj_country['ec_max_contribution'], errors='coerce'
).fillna(0.0)

# STEP 4: Total contribution per country (summing contributions of all projects linked to that country)
total_contrib_per_country = proj_country.groupby('country')['ec_max_contribution'].sum().reset_index()
total_contrib_per_country.rename(columns={'ec_max_contribution': 'total_contribution'}, inplace=True)

# STEP 5: Count number of unique projects per country
project_count_per_country = proj_country.groupby('country')['id'].nunique().reset_index()
project_count_per_country.rename(columns={'id': 'project_count'}, inplace=True)

# STEP 6: Merge both into one country-specific DataFrame
country_specific = pd.merge(
    total_contrib_per_country,
    project_count_per_country,
    on='country',
    how='outer'
)

In [81]:
# Count number of unique projects per country
country_counts = proj_country['country'].value_counts().reset_index()
country_counts.columns = ['country', 'project_count']

country_counts

,country,project_count
0,DE,5876
1,ES,4940
2,IT,4622
3,FR,4462
4,NL,3711
...,...,...
166,SV,1
167,SR,1
168,NC,1
169,NI,1


In [55]:
country_specific

,country,total_contribution,project_count
0,,16876536.0,13
1,AD,0.0,1
2,AE,8269188.0,5
3,AF,3392025.0,1
4,AI,0.0,1
...,...,...,...
166,VN,25416205.0,15
167,XK,27997780.0,14
168,ZA,353035252.0,143
169,ZM,38528631.0,20


In [60]:
country_specific['total_contribution'] = pd.to_numeric(
    country_specific['total_contribution'], errors='coerce').fillna(0.0)
country_specific['project_count'] = pd.to_numeric(
    country_specific['project_count'], errors='coerce').fillna(0.0)

# Step 2: Compute € per 100k inhabitants (from the dictionary)
def compute_euro_per_100k(row):
    pop = country_populations.get(row['country'])
    if pop is None or pop == 0:
        return None
    return row['total_contribution'] / pop * 100_000

In [64]:
country_specific['€/100k_inhabitants'] = country_specific.apply(compute_euro_per_100k, axis=1)
country_specific['€/100k_inhabitants'] = country_specific['€/100k_inhabitants'].fillna(0.0)
country_specific

,country,total_contribution,project_count,€/100k_inhabitants
0,,16876536.0,13,0.000000e+00
1,AD,0.0,1,0.000000e+00
2,AE,8269188.0,5,0.000000e+00
3,AF,3392025.0,1,8.454371e+03
4,AI,0.0,1,0.000000e+00
...,...,...,...,...
166,VN,25416205.0,15,2.403220e+04
167,XK,27997780.0,14,1.416108e+06
168,ZA,353035252.0,143,5.840830e+05
169,ZM,38528631.0,20,1.852417e+05


In [66]:
import pycountry

def iso2_to_iso3(iso2):
    try:
        return pycountry.countries.get(alpha_2=iso2).alpha_3
    except:
        return None

country_specific['iso_alpha_3'] = country_specific['country'].apply(iso2_to_iso3)

In [69]:
print(country_specific[['iso_alpha_3', 'total_contribution']].describe())

       total_contribution
count        1.710000e+02
mean         8.047449e+08
std          1.924296e+09
min          0.000000e+00
25%          4.298382e+06
50%          2.077420e+07
75%          2.388524e+08
max          1.146752e+10


In [70]:
# Replace non-positive values (e.g. 0 or negative) with NaN to avoid log10 errors
country_specific['log_€/100k_inhabitants'] = np.log10(
    country_specific['€/100k_inhabitants'].replace(0, np.nan)
)

In [72]:

fig = px.scatter_geo(
    country_specific,
    locations="iso_alpha_3",
    locationmode="ISO-3",
    size="project_count",
    color="log_€/100k_inhabitants",
    hover_name="country",
    showcountries=True,
    hover_data={
        "total_contribution": ':.0f',
        "project_count": True,
        "€/100k_inhabitants": ':.2f',
        "log_€/100k_inhabitants": ':.2f',
        "iso_alpha_3": False
    },
    projection="natural earth",
    color_continuous_scale="Viridis",
    title="EU Horizon Projects: EC Contribution per Country (€/100k inhabitants, Log Scale)"
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))

fig.show()

In [74]:
country_centroids_iso3 = {
    'MDG': (-18.766947, 46.869107), 'SEN': (14.497401, -14.452362), 'ETH': (9.145, 40.489673),
    'GBR': (55.378051, -3.435973), 'CHE': (46.818188, 8.227512), 'FRA': (46.603354, 1.888334),
    'AUS': (-25.274398, 133.775136), 'FIN': (61.92411, 25.748151), 'DNK': (56.26392, 9.501785),
    'ESP': (40.463667, -3.74922), 'SVN': (46.151241, 14.995463), 'LTU': (55.169438, 23.881275),
    'POL': (51.919438, 19.145136), 'NLD': (52.132633, 5.291266), 'PRT': (39.399872, -8.224454),
    'BEL': (50.503887, 4.469936), 'DEU': (51.165691, 10.451526), 'USA': (37.09024, -95.712891),
    'NOR': (60.472024, 8.468946), 'TUR': (38.963745, 35.243322), 'ZAF': (-30.559482, 22.937506),
    'ZMB': (-13.133897, 27.849332), 'ZWE': (-19.015438, 29.154857), 'CIV': (7.539989, -5.54708),
    'SVK': (48.669026, 19.699024), 'BGR': (42.733883, 25.48583), 'ROU': (45.943161, 24.96676),
    'GRC': (39.074208, 21.824312), 'ISR': (31.046051, 34.851612), 'ITA': (41.87194, 12.56738),
    'EST': (58.595272, 25.013607), 'IRL': (53.41291, -8.24389), 'HUN': (47.162494, 19.503304),
    'CZE': (49.817492, 15.472962), 'AUT': (47.516231, 14.550072), 'LVA': (56.879635, 24.603189),
    'UKR': (48.379433, 31.16558), 'GIN': (9.945587, -9.696645), 'MLI': (17.570692, -3.996166),
    'SWE': (60.128161, 18.643501), 'BWA': (-22.328474, 24.684866), 'MOZ': (-18.665695, 35.529562),
    'LSO': (-29.609988, 28.233608), 'SWZ': (-26.522503, 31.465866), 'BFA': (12.238333, -1.561593),
    'GHA': (7.946527, -1.023194), 'CYP': (35.126413, 33.429859), 'MLT': (35.937496, 14.375416),
    'CMR': (7.369722, 12.354722), 'LUX': (49.815273, 6.129583), 'NGA': (9.081999, 8.675277),
    'TZA': (-6.369028, 34.888822), 'MWI': (-13.254308, 34.301525), 'UGA': (1.373333, 32.290275),
    'KEN': (-0.023559, 37.906193), 'CHN': (35.86166, 104.195397), 'IND': (20.593684, 78.96288),
    'KOR': (35.907757, 127.766922), 'SRB': (44.016521, 21.005859), 'EGY': (26.820553, 30.802498),
    'ARG': (-38.416097, -63.616672), 'HRV': (45.1, 15.2), 'ARM': (40.069099, 45.038189),
    'BRA': (-14.235004, -51.92528), 'CPV': (16.5388, -23.0418), 'CAN': (56.130366, -106.346771),
    'TUN': (33.886917, 9.537499), 'AGO': (-11.202692, 17.873887), 'STP': (0.18636, 6.613081),
    'COL': (4.570868, -74.297333), 'BTN': (27.514162, 90.433601), 'PRY': (-23.442503, -58.443832),
    'CAF': (6.611111, 20.939444), 'DZA': (28.033886, 1.659626), 'GNQ': (1.650801, 10.267895),
    'LKA': (7.873054, 80.771797), 'CHL': (-35.675147, -71.542969), 'ALB': (41.153332, 20.168331),
    'ISL': (64.963051, -19.020835), 'COD': (-4.038333, 21.758664), 'BDI': (-3.373056, 29.918886),
    'MEX': (23.634501, -102.552784), 'MNE': (42.708678, 19.37439), 'MNG': (46.862496, 103.846656),
    'THA': (15.870032, 100.992541), 'KAZ': (48.019573, 66.923684), 'JPN': (36.204824, 138.252924),
    'VAT': (41.902916, 12.453389), 'NZL': (-40.900557, 174.885971), 'ECU': (-1.831239, -78.183406),
    'MDA': (47.411631, 28.369885), 'UZB': (41.377491, 64.585262), 'AZE': (40.143105, 47.576927),
    'SGP': (1.352083, 103.819836), 'PAK': (30.375321, 69.345116), 'TWN': (23.69781, 120.960515),
    'GUM': (13.444304, 144.793731), 'CRI': (9.748917, -83.753428), 'PER': (-9.189967, -75.015152),
    'LBN': (33.854721, 35.862285), 'BIH': (43.915886, 17.679076), 'MAR': (31.791702, -7.09262),
    'VNM': (14.058324, 108.277199), 'MKD': (41.608635, 21.745275), 'BEN': (9.30769, 2.315834),
    'GAB': (-0.803689, 11.609444), 'MYS': (4.210484, 101.975766), 'XKX': (42.602636, 20.902977),
    'PSE': (31.952162, 35.233154), 'PHL': (12.879721, 121.774017), 'SAU': (23.885942, 45.079162),
    'RWA': (-1.940278, 29.873888), 'IDN': (-0.789275, 113.921327), 'FRO': (61.892635, -6.911806),
    'CUB': (21.521757, -77.781167), 'KGZ': (41.20438, 74.766098), 'BGD': (23.684994, 90.356331),
    'PYF': (-17.679742, -149.406843), 'LBR': (6.428055, -9.429499), 'SLE': (8.460555, -11.779889),
    'VEN': (6.42375, -66.58973), 'GEO': (42.315407, 43.356892), 'JOR': (30.585164, 36.238414),
    'FJI': (-17.713371, 178.065032), 'URY': (-32.522779, -55.765835), 'COG': (-0.228021, 15.827659),
    'AFG': (33.93911, 67.709953), 'IRQ': (33.223191, 43.679291), 'HKG': (22.396428, 114.109497),
    'TJK': (38.861034, 71.276093), 'TKM': (38.969719, 59.556278), 'BOL': (-16.290154, -63.588653),
    'MDV': (3.202778, 73.22068), 'IMN': (54.236107, -4.548056), 'BRB': (13.193887, -59.543198),
    'BHR': (25.930414, 50.637772), 'GRL': (71.706936, -42.604303), 'GNQ': (1.650801, 10.267895),
    'DMA': (15.415, -61.371), 'MHL': (7.1315, 171.1845),
}


In [75]:
country_specific['lat'] = country_specific['iso_alpha_3'].map(lambda c: country_centroids_iso3.get(c, (None, None))[0])
country_specific['lon'] = country_specific['iso_alpha_3'].map(lambda c: country_centroids_iso3.get(c, (None, None))[1])


In [79]:
fig = px.scatter_map(
    country_specific,
    lat='lat',    # you need lat/lon columns for the centers of countries
    lon='lon',
    size='project_count',
    color='log_€/100k_inhabitants',
    hover_name='country',
    zoom=1.5,
    map_style="open-street-map"
)

fig.show()

In [86]:
# Parse geolocation into lat/lon columns
df_org[['latitude', 'longitude']] = df_org['geolocation'].str.split(',', expand=True).astype(float)


ValueError: could not convert string to float: ''

In [85]:
# add interactive options
interact(
    plot_filtered_map,
    metric = widgets.Dropdown(options=[
        'Total Contribution', 'Total Cost', 'Projects per Country', '€/100k inhabitants'
    ]),
    funding_scheme = widgets.Dropdown(
        options=[None] + sorted(df_proj['funding_scheme'].dropna().unique())
    ),
    field = widgets.Dropdown(
        options=[None] + sorted(df_proj['field'].dropna().unique())
    ),
    sub_field = widgets.Dropdown(
        options=[None] + sorted(df_proj['sub_field'].dropna().unique())
                                ),
    niche = widgets.Dropdown(
        options=[None] + sorted(df_proj['niche'].dropna().unique())
                            ),
    min_total_cost = widgets.FloatSlider(    # TODO: add min and max based on dataFrame information
        min=0, max=1e7, step=1e5, value=0
                                        ),
    max_total_cost = widgets.FloatSlider(    # TODO: add min and max based on dataFrame information
        min=1e6, max=1e9, step=1e7, value=1e9
                                        ),
    start_year = widgets.IntRangeSlider(     # TODO: add min and max based on dataFrame information    
        min=2014, max=2025, value=(2021, 2025), step=1
                                       )
);

NameError: name 'plot_filtered_map' is not defined